<a href="https://colab.research.google.com/github/snsnsx/128/blob/main/Transcribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#======== Скачать с локального носителя =======================
from pydub import AudioSegment
import os
from pathlib import Path
from google.colab import files

uploaded = files.upload()
audio_path = next(iter(uploaded))
print("Файл загружен:", audio_path)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Saving Новая запись 7.m4a to Новая запись 7.m4a
Файл загружен: Новая запись 7.m4a


In [ ]:
# =============== Загрузка библиотек ===============
%%capture
import subprocess
subprocess.run(["pip", "install", "-q", "openai-whisper"])
import whisper
model = whisper.load_model(name="large-v3-turbo", in_memory=True)

In [ ]:
import subprocess

# ======= Оптимизация файла =====================
def optimize_audio(input_file, output_file):
    subprocess.run([
        "ffmpeg",
        "-i", input_file,
        "-vn",                 # убрать видео
        "-ac", "1",            # моно
        "-ar", "16000",        # 16 кГц
        "-c:a", "libopus",     # кодек Opus
        "-b:a", "24k",         # битрейт
        "-y",
        output_file
    ], check=True)

optimize_audio(audio_path, "optimized.opus")


# ======== Транскрибация =========================
chunk_length_ms = 10 * 60 * 1000  # 10 минут
audio = AudioSegment.from_file(audio_path)
results = []
for i, start in enumerate(range(0, len(audio), chunk_length_ms)):
    end = min(start + chunk_length_ms, len(audio))
    chunk = audio[start:end]
    chunk_file = f"chunk_{i:03d}.wav"
    chunk.export(chunk_file, format="wav")
    result = model.transcribe(
        audio=chunk_file,
        verbose=True,
        fp16=False,
        initial_prompt=None,
        language="ru",
    )
    results.append(result["text"])
    os.remove(chunk_file)
full_text = "\n".join(results)

[00:00.000 --> 00:06.600]  Давайте на одном примере посмотрим, как можно быстро делить многочлены.
[00:06.600 --> 00:12.900]  Сначала определимся со степенью результата деления. Она будет равна разности старших
[00:12.900 --> 00:18.780]  степеней числителей и знаменателей. Как здесь. Кубический многочлен числителя – старшая степень
[00:18.780 --> 00:24.180]  тройка. Линейный знаменатель – старшая степень единица. Степень итогового многочлена – 2.
[00:24.180 --> 00:31.560]  Запишем результат деления в общем виде. Теперь быстро найдем неизвестные коэффициенты. Все скобки
[00:31.560 --> 00:36.360]  раскрывать не будем. Заметим, что вклад в коэффициент при степени х вносит только произведение
[00:36.360 --> 00:43.900]  двух больших степеней внутри скобок. Х кубе равно ах. А равно единице. Аналогично правой части вклад
[00:43.900 --> 00:50.460]  свободный член вносит только произведение двух меньших степеней. 1 равно 1 на с. Значит,
[00:50.460 --> 00:57.180]  c единица. Осталось найти b. Да

In [ ]:
# ============ AI-Причёсывание ===================
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI')

from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)

promt = "Вот запись конспекта занятия. Причеши и структурируй. В ответ пришли только готовую версию"
interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input=promt+full_text
)
full_text = interaction.output_text


TimeoutException: Requesting secret GEMINI timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# ========= Отправка. mail + Telegram
sender_email = "snsnsx@yandex.ru"
from google.colab import userdata
password = userdata.get("YANDEX")

import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

receiver_email = "pnpnsx@gmail.com"
msg = MIMEMultipart()
msg["From"] = sender_email
msg["To"] = receiver_email
msg["Subject"] = "Транскрибация"
msg.attach(MIMEText(full_text))

with smtplib.SMTP("smtp.yandex.ru", 587) as server:
    server.starttls()
    server.login(sender_email, password)
    server.send_message(msg)

print("Письмо отправлено")

#subprocess.run(["pip", "install", "python-telegram-bot"])
import requests

TOKEN = userdata.get("TELEGRAM")
CHAT_ID = userdata.get("CHAT_ID")

requests.post(
    f"https://api.telegram.org/bot{TOKEN}/sendMessage",
    json={
        "chat_id": CHAT_ID,
        "text": full_text,
    }
)

In [ ]:
# ======= Прямое скачивание ==========================
with open ("file.txt", "w") as f:
  f.write(full_text)
files.download("file.txt")

In [ ]:
if False:
  import requests
  response = requests.get(f"https://api.telegram.org/bot{TOKEN}/getUpdates")
  print(response.status_code)
  print(response.text)